# TPOSE24: Boundary Discontinuity Videos

Composite view: TPOSE6 as background (5° buffer surrounding TPOSE24), TPOSE24 filled inside its domain.
A dashed black box marks the TPOSE24 boundary so discontinuities are visible.

Variables: SST, SSH, surface U/V, 100 m U/V, 500 m U/V  
TPOSE24: 6-hourly frames (120 total). TPOSE6: updates daily.  
Videos saved as HTML with a play bar (open in any browser).

In [1]:
import os
os.makedirs('surface', exist_ok=True)
os.makedirs('velocity', exist_ok=True)
os.makedirs('TS', exist_ok=True)
import numpy as np
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import matplotlib.cm as mcm
import matplotlib.colors as mcolors
import cmocean.cm as cmo
from xmitgcm import open_mdsdataset
import warnings
warnings.filterwarnings('ignore')

# Point matplotlib at the bundled ffmpeg binary
import imageio_ffmpeg
matplotlib.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
_writer = animation.FFMpegWriter(fps=6, codec='h264',
                                  extra_args=['-pix_fmt', 'yuv420p'])
print('ffmpeg:', matplotlib.rcParams['animation.ffmpeg_path'])

def make_levels(data_arr, symmetric=False, plo=1, phi=99):
    """51 contourf levels. symmetric=True for velocity/SSH diverging colormaps."""
    if symmetric:
        vmax = float(np.nanpercentile(np.abs(data_arr), phi))
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1e-10
        return np.linspace(-vmax, vmax, 51)
    else:
        vmin = float(np.nanpercentile(data_arr[np.isfinite(data_arr)], plo))
        vmax = float(np.nanpercentile(data_arr[np.isfinite(data_arr)], phi))
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
            vmin, vmax = -1e-10, 1e-10
        return np.linspace(vmin, vmax, 51)

def add_colorbar(fig, ax, levels, cmap, label):
    norm = mcolors.Normalize(vmin=levels[0], vmax=levels[-1])
    sm = mcm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    return fig.colorbar(sm, ax=ax, label=label, shrink=0.85)

def draw_composite(ax, x6, y6, arr6, x24, y24, arr24, levels, cmap,
                   lon_min24, lon_max24, lat_min24, lat_max24):
    """Plot TPOSE6 background then TPOSE24 overlay with dashed boundary."""
    ax.contourf(x6, y6, arr6, levels=levels, cmap=cmap, extend='both')
    ax.contourf(x24, y24, arr24, levels=levels, cmap=cmap, extend='both')
    ax.add_patch(mpatches.Rectangle(
        (lon_min24, lat_min24), lon_max24 - lon_min24, lat_max24 - lat_min24,
        lw=0.8, edgecolor='black', facecolor='none', ls='--', zorder=10,
    ))

def clear_ax(ax):
    while ax.collections:
        ax.collections[0].remove()
    for p in list(ax.patches):
        p.remove()

ffmpeg: /home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/imageio_ffmpeg/binaries/ffmpeg-linux-x86_64-v7.0.2


## 1. Load TPOSE24 (lazy)

In [2]:
run24 = '/data/SO3/edavenport/tpose24/oct2012_TP6Vel'
ds24 = open_mdsdataset(
    data_dir=run24, grid_dir=run24,
    iters=list(range(12, 8641, 12)),
    prefix=['diag_state', 'diag_surf'],
    ref_date='2012-10-01', delta_t=300,
)
for c in ('XC', 'YC', 'Z', 'Zl', 'XG', 'YG'):
    if c in ds24.coords:
        ds24[c] = ds24[c].astype(float)

lon_min24 = float(ds24.XC.min())
lon_max24 = float(ds24.XC.max())
lat_min24 = float(ds24.YC.min())
lat_max24 = float(ds24.YC.max())

# 6-hourly subset (120 frames for Oct)
ds24_6h = ds24.isel(time=slice(None, None, 6))
t24_times = ds24_6h.time.values

print(f'TPOSE24: {lon_min24:.2f}–{lon_max24:.2f}°E, {lat_min24:.2f}–{lat_max24:.2f}°N')
print(f'6-hourly frames: {len(t24_times)}')

TPOSE24: 209.35–230.65°E, -5.48–10.48°N
6-hourly frames: 120


## 2. Load TPOSE6 (lazy) — buffer region, Oct 2012

In [3]:
buf_deg = 5.0
buf_lon_min = lon_min24 - buf_deg
buf_lon_max = lon_max24 + buf_deg
buf_lat_min = lat_min24 - buf_deg
buf_lat_max = lat_max24 + buf_deg

run6  = '/data/SO3/edavenport/tpose6/sep2012/run_iter14'
grid6 = '/data/SO6/TPOSE_diags/tpose6/grid_6'
ds6 = open_mdsdataset(
    data_dir=run6, grid_dir=grid6,
    iters=list(range(72, 8785, 72)),
    prefix=['diag_state', 'diag_surf'],
    ref_date='2012-09-01', delta_t=1200,
)
for c in ('XC', 'YC', 'Z', 'XG', 'YG'):
    if c in ds6.coords:
        ds6[c] = ds6[c].astype(float)

ds6_buf = ds6.sel(
    XC=slice(buf_lon_min, buf_lon_max),
    YC=slice(buf_lat_min, buf_lat_max),
    time=slice('2012-10-01', '2012-10-31'),
)
t6_times = ds6_buf.time.values

print(f'TPOSE6 buffer: {float(ds6_buf.XC.min()):.2f}–{float(ds6_buf.XC.max()):.2f}°E, '
      f'{float(ds6_buf.YC.min()):.2f}–{float(ds6_buf.YC.max()):.2f}°N')
print(f'TPOSE6 daily steps: {len(t6_times)}')

# Index into TPOSE6 daily for a given TPOSE24 frame
def t6_idx(i24):
    idx = int(np.searchsorted(t6_times, t24_times[i24], side='right') - 1)
    return max(0, min(idx, len(t6_times) - 1))

TPOSE6 buffer: 204.42–235.58°E, -10.42–15.42°N
TPOSE6 daily steps: 31


## 3. Pre-compute TPOSE6 surface and depth fields

In [4]:
print('Computing TPOSE6 fields...')
z6_sfc = float(ds6_buf.Z.sel(Z=0, method='nearest'))

sst6   = ds6_buf.THETA.sel(Z=z6_sfc, method='nearest').compute()
ssh6   = ds6_buf.ETAN.compute()
u6_sfc = ds6_buf.UVEL.sel(Z=z6_sfc, method='nearest').compute()
v6_sfc = ds6_buf.VVEL.sel(Z=z6_sfc, method='nearest').compute()
u6_100 = ds6_buf.UVEL.sel(Z=-100., method='nearest').compute()
v6_100 = ds6_buf.VVEL.sel(Z=-100., method='nearest').compute()
u6_500 = ds6_buf.UVEL.sel(Z=-500., method='nearest').compute()
v6_500 = ds6_buf.VVEL.sel(Z=-500., method='nearest').compute()

# WVEL is on Zl (layer-top interface) grid
if 'Zl' in ds6_buf.coords:
    ds6_buf['Zl'] = ds6_buf['Zl'].astype(float)
zl6_sfc = float(ds6_buf.Zl.sel(Zl=0,     method='nearest'))
zl6_100 = float(ds6_buf.Zl.sel(Zl=-100., method='nearest'))
zl6_500 = float(ds6_buf.Zl.sel(Zl=-500., method='nearest'))
w6_sfc = ds6_buf.WVEL.sel(Zl=zl6_sfc, method='nearest').compute()
w6_100 = ds6_buf.WVEL.sel(Zl=zl6_100, method='nearest').compute()
w6_500 = ds6_buf.WVEL.sel(Zl=zl6_500, method='nearest').compute()

xc6 = ds6_buf.XC.values
yc6 = ds6_buf.YC.values
xg6 = ds6_buf.XG.values
yg6 = ds6_buf.YG.values

print('TPOSE6 done.')

Computing TPOSE6 fields...


TPOSE6 done.


## 4. Snapshot helper

Picks Oct 15 (frame index ~56 at 6-hourly).

In [5]:
# Oct 15 snapshot frame
snap_t24 = np.datetime64('2012-10-15T12:00')
snap_i24 = int(np.argmin(np.abs(t24_times - snap_t24)))
snap_i6  = t6_idx(snap_i24)
print(f'Snapshot: TPOSE24 frame {snap_i24} ({t24_times[snap_i24]}), '
      f'TPOSE6 frame {snap_i6} ({t6_times[snap_i6]})')

Snapshot: TPOSE24 frame 58 (2012-10-15T13:00:00.000000000), TPOSE6 frame 14 (2012-10-15T00:00:00.000000000)


## 5. SST

In [6]:
print('Loading TPOSE24 SST...')
z24_sfc = float(ds24.Z.sel(Z=0, method='nearest'))
sst24 = ds24_6h.THETA.sel(Z=z24_sfc, method='nearest').compute()
xc24  = ds24.XC.values
yc24  = ds24.YC.values

# Shared color levels from both models
combined_sst = np.concatenate([sst24.values.ravel(), sst6.values.ravel()])
sst_levels = make_levels(combined_sst, symmetric=False)
cmap_sst = cmo.thermal

# --- Snapshot ---
fig, ax = plt.subplots(figsize=(11, 6))
draw_composite(ax, xc6, yc6, sst6.isel(time=snap_i6).values,
               xc24, yc24, sst24.isel(time=snap_i24).values,
               sst_levels, cmap_sst,
               lon_min24, lon_max24, lat_min24, lat_max24)
add_colorbar(fig, ax, sst_levels, cmap_sst, 'SST (°C)')
ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title(f'SST: TPOSE6 background + TPOSE24 fill\n{str(t24_times[snap_i24])[:13]}  '
             '(dashed = TPOSE24 boundary)')
plt.tight_layout()
plt.savefig('surface/snapshot_SST.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved snapshot_SST.png')

# --- Animation ---
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(buf_lon_min, buf_lon_max)
ax.set_ylim(buf_lat_min, buf_lat_max)
ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
add_colorbar(fig, ax, sst_levels, cmap_sst, 'SST (°C)')
title_txt = ax.set_title('')
draw_composite(ax, xc6, yc6, sst6.isel(time=0).values,
               xc24, yc24, sst24.isel(time=0).values,
               sst_levels, cmap_sst,
               lon_min24, lon_max24, lat_min24, lat_max24)
plt.tight_layout()

def draw_sst(i):
    clear_ax(ax)
    draw_composite(ax, xc6, yc6, sst6.isel(time=t6_idx(i)).values,
                   xc24, yc24, sst24.isel(time=i).values,
                   sst_levels, cmap_sst,
                   lon_min24, lon_max24, lat_min24, lat_max24)
    title_txt.set_text(f'SST  {str(t24_times[i])[:13]}  '
                       f'[TP6: {str(t6_times[t6_idx(i)])[:10]}]')

anim = animation.FuncAnimation(fig, draw_sst, frames=len(t24_times), interval=150)
print('Rendering SST video...')
anim.save('surface/video_SST.mp4', writer=_writer, dpi=100)
plt.close()
print('Saved video_SST.mp4')
del sst24

Loading TPOSE24 SST...


Saved snapshot_SST.png
Rendering SST video...


Saved video_SST.mp4


## 6. SSH

In [7]:
print('Loading TPOSE24 SSH...')
ssh24 = ds24_6h.ETAN.compute()

combined_ssh = np.concatenate([ssh24.values.ravel(), ssh6.values.ravel()])
ssh_levels = make_levels(combined_ssh, symmetric=True)
cmap_ssh = cmo.balance

# --- Snapshot ---
fig, ax = plt.subplots(figsize=(11, 6))
draw_composite(ax, xc6, yc6, ssh6.isel(time=snap_i6).values,
               xc24, yc24, ssh24.isel(time=snap_i24).values,
               ssh_levels, cmap_ssh,
               lon_min24, lon_max24, lat_min24, lat_max24)
add_colorbar(fig, ax, ssh_levels, cmap_ssh, 'SSH (m)')
ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title(f'SSH: TPOSE6 background + TPOSE24 fill\n{str(t24_times[snap_i24])[:13]}  '
             '(dashed = TPOSE24 boundary)')
plt.tight_layout()
plt.savefig('surface/snapshot_SSH.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved snapshot_SSH.png')

# --- Animation ---
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(buf_lon_min, buf_lon_max)
ax.set_ylim(buf_lat_min, buf_lat_max)
ax.axhline(0, color='gray', lw=0.5, ls=':')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
add_colorbar(fig, ax, ssh_levels, cmap_ssh, 'SSH (m)')
title_txt = ax.set_title('')
draw_composite(ax, xc6, yc6, ssh6.isel(time=0).values,
               xc24, yc24, ssh24.isel(time=0).values,
               ssh_levels, cmap_ssh,
               lon_min24, lon_max24, lat_min24, lat_max24)
plt.tight_layout()

def draw_ssh(i):
    clear_ax(ax)
    draw_composite(ax, xc6, yc6, ssh6.isel(time=t6_idx(i)).values,
                   xc24, yc24, ssh24.isel(time=i).values,
                   ssh_levels, cmap_ssh,
                   lon_min24, lon_max24, lat_min24, lat_max24)
    title_txt.set_text(f'SSH  {str(t24_times[i])[:13]}  '
                       f'[TP6: {str(t6_times[t6_idx(i)])[:10]}]')

anim = animation.FuncAnimation(fig, draw_ssh, frames=len(t24_times), interval=150)
print('Rendering SSH video...')
anim.save('surface/video_SSH.mp4', writer=_writer, dpi=100)
plt.close()
print('Saved video_SSH.mp4')
del ssh24

Loading TPOSE24 SSH...


Saved snapshot_SSH.png
Rendering SSH video...


Saved video_SSH.mp4


## 7. Surface velocity (U and V)

In [8]:
print('Loading TPOSE24 surface U/V/W...')
u24_sfc = ds24_6h.UVEL.sel(Z=z24_sfc, method='nearest').compute()
v24_sfc = ds24_6h.VVEL.sel(Z=z24_sfc, method='nearest').compute()
xg24  = ds24.XG.values
yg24  = ds24.YG.values

# WVEL on Zl grid
zl24_sfc = float(ds24.Zl.sel(Zl=0, method='nearest'))
w24_sfc = ds24_6h.WVEL.sel(Zl=zl24_sfc, method='nearest').compute()

combined_u = np.concatenate([u24_sfc.values.ravel(), u6_sfc.values.ravel()])
combined_v = np.concatenate([v24_sfc.values.ravel(), v6_sfc.values.ravel()])
combined_w = np.concatenate([w24_sfc.values.ravel(), w6_sfc.values.ravel()])
u_sfc_levels = make_levels(combined_u, symmetric=True)
v_sfc_levels = make_levels(combined_v, symmetric=True)
w_sfc_levels = make_levels(combined_w, symmetric=True)
cmap_vel = cmo.balance

z24_sfc_actual = float(ds24.Z.sel(Z=z24_sfc, method='nearest'))

# --- Snapshot ---
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
panels = [
    (axes[0], u24_sfc.isel(time=snap_i24).values, u6_sfc.isel(time=snap_i6).values,
     xg24, yc24, xg6, yc6, u_sfc_levels, f'UVEL (m/s) sfc Z≈{z24_sfc_actual:.0f}m'),
    (axes[1], v24_sfc.isel(time=snap_i24).values, v6_sfc.isel(time=snap_i6).values,
     xc24, yg24, xc6, yg6, v_sfc_levels, f'VVEL (m/s) sfc Z≈{z24_sfc_actual:.0f}m'),
    (axes[2], w24_sfc.isel(time=snap_i24).values, w6_sfc.isel(time=snap_i6).values,
     xc24, yc24, xc6, yc6, w_sfc_levels, f'WVEL (m/s) Zl≈{zl24_sfc:.0f}m'),
]
for ax, arr24, arr6, x24, y24, x6, y6, levels, label in panels:
    draw_composite(ax, x6, y6, arr6, x24, y24, arr24, levels, cmap_vel,
                   lon_min24, lon_max24, lat_min24, lat_max24)
    add_colorbar(fig, ax, levels, cmap_vel, label)
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
axes[0].set_title(f'Surface U  {str(t24_times[snap_i24])[:13]}')
axes[1].set_title(f'Surface V  {str(t24_times[snap_i24])[:13]}')
axes[2].set_title(f'Surface W  {str(t24_times[snap_i24])[:13]}')
fig.suptitle('Surface velocity: TPOSE6 background + TPOSE24 fill  (dashed = TP24 boundary)', fontsize=12)
plt.tight_layout()
plt.savefig('velocity/snapshot_velocity_surface.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved snapshot_velocity_surface.png')

# --- Animation ---
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for ax in axes:
    ax.set_xlim(buf_lon_min, buf_lon_max)
    ax.set_ylim(buf_lat_min, buf_lat_max)
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
add_colorbar(fig, axes[0], u_sfc_levels, cmap_vel, 'UVEL (m/s)')
add_colorbar(fig, axes[1], v_sfc_levels, cmap_vel, 'VVEL (m/s)')
add_colorbar(fig, axes[2], w_sfc_levels, cmap_vel, 'WVEL (m/s)')
title_txt = fig.suptitle('', fontsize=11)
draw_composite(axes[0], xg6, yc6, u6_sfc.isel(time=0).values,
               xg24, yc24, u24_sfc.isel(time=0).values,
               u_sfc_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
draw_composite(axes[1], xc6, yg6, v6_sfc.isel(time=0).values,
               xc24, yg24, v24_sfc.isel(time=0).values,
               v_sfc_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
draw_composite(axes[2], xc6, yc6, w6_sfc.isel(time=0).values,
               xc24, yc24, w24_sfc.isel(time=0).values,
               w_sfc_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
plt.tight_layout()

def draw_vel_sfc(i):
    clear_ax(axes[0]); clear_ax(axes[1]); clear_ax(axes[2])
    i6 = t6_idx(i)
    draw_composite(axes[0], xg6, yc6, u6_sfc.isel(time=i6).values,
                   xg24, yc24, u24_sfc.isel(time=i).values,
                   u_sfc_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
    draw_composite(axes[1], xc6, yg6, v6_sfc.isel(time=i6).values,
                   xc24, yg24, v24_sfc.isel(time=i).values,
                   v_sfc_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
    draw_composite(axes[2], xc6, yc6, w6_sfc.isel(time=i6).values,
                   xc24, yc24, w24_sfc.isel(time=i).values,
                   w_sfc_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
    title_txt.set_text(f'Surface U/V/W  {str(t24_times[i])[:13]}  '
                       f'[TP6: {str(t6_times[i6])[:10]}]')

anim = animation.FuncAnimation(fig, draw_vel_sfc, frames=len(t24_times), interval=150)
print('Rendering surface velocity video...')
anim.save('velocity/video_velocity_surface.mp4', writer=_writer, dpi=100)
plt.close()
print('Saved video_velocity_surface.mp4')
del u24_sfc, v24_sfc, w24_sfc

Loading TPOSE24 surface U/V/W...


Saved snapshot_velocity_surface.png


Rendering surface velocity video...


Saved video_velocity_surface.mp4


## 8. Velocity at 100 m

In [9]:
print('Loading TPOSE24 100 m U/V/W...')
z_100 = float(ds24.Z.sel(Z=-100., method='nearest'))
u24_100 = ds24_6h.UVEL.sel(Z=z_100, method='nearest').compute()
v24_100 = ds24_6h.VVEL.sel(Z=z_100, method='nearest').compute()

zl24_100 = float(ds24.Zl.sel(Zl=-100., method='nearest'))
w24_100 = ds24_6h.WVEL.sel(Zl=zl24_100, method='nearest').compute()

combined_u100 = np.concatenate([u24_100.values.ravel(), u6_100.values.ravel()])
combined_v100 = np.concatenate([v24_100.values.ravel(), v6_100.values.ravel()])
combined_w100 = np.concatenate([w24_100.values.ravel(), w6_100.values.ravel()])
u100_levels = make_levels(combined_u100, symmetric=True)
v100_levels = make_levels(combined_v100, symmetric=True)
w100_levels = make_levels(combined_w100, symmetric=True)

# --- Snapshot ---
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
panels = [
    (axes[0], u24_100.isel(time=snap_i24).values, u6_100.isel(time=snap_i6).values,
     xg24, yc24, xg6, yc6, u100_levels, f'UVEL (m/s) Z≈{z_100:.0f}m'),
    (axes[1], v24_100.isel(time=snap_i24).values, v6_100.isel(time=snap_i6).values,
     xc24, yg24, xc6, yg6, v100_levels, f'VVEL (m/s) Z≈{z_100:.0f}m'),
    (axes[2], w24_100.isel(time=snap_i24).values, w6_100.isel(time=snap_i6).values,
     xc24, yc24, xc6, yc6, w100_levels, f'WVEL (m/s) Zl≈{zl24_100:.0f}m'),
]
for ax, arr24, arr6, x24, y24, x6, y6, levels, label in panels:
    draw_composite(ax, x6, y6, arr6, x24, y24, arr24, levels, cmap_vel,
                   lon_min24, lon_max24, lat_min24, lat_max24)
    add_colorbar(fig, ax, levels, cmap_vel, label)
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
axes[0].set_title(f'U at {z_100:.0f} m  {str(t24_times[snap_i24])[:13]}')
axes[1].set_title(f'V at {z_100:.0f} m  {str(t24_times[snap_i24])[:13]}')
axes[2].set_title(f'W at Zl≈{zl24_100:.0f} m  {str(t24_times[snap_i24])[:13]}')
fig.suptitle(f'Velocity at Z≈{z_100:.0f}m: TPOSE6 background + TPOSE24 fill', fontsize=12)
plt.tight_layout()
plt.savefig('velocity/snapshot_velocity_100m.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved snapshot_velocity_100m.png')

# --- Animation ---
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for ax in axes:
    ax.set_xlim(buf_lon_min, buf_lon_max)
    ax.set_ylim(buf_lat_min, buf_lat_max)
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
add_colorbar(fig, axes[0], u100_levels, cmap_vel, f'UVEL (m/s) {z_100:.0f}m')
add_colorbar(fig, axes[1], v100_levels, cmap_vel, f'VVEL (m/s) {z_100:.0f}m')
add_colorbar(fig, axes[2], w100_levels, cmap_vel, f'WVEL (m/s) {zl24_100:.0f}m')
title_txt = fig.suptitle('', fontsize=11)
draw_composite(axes[0], xg6, yc6, u6_100.isel(time=0).values,
               xg24, yc24, u24_100.isel(time=0).values,
               u100_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
draw_composite(axes[1], xc6, yg6, v6_100.isel(time=0).values,
               xc24, yg24, v24_100.isel(time=0).values,
               v100_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
draw_composite(axes[2], xc6, yc6, w6_100.isel(time=0).values,
               xc24, yc24, w24_100.isel(time=0).values,
               w100_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
plt.tight_layout()

def draw_vel_100(i):
    clear_ax(axes[0]); clear_ax(axes[1]); clear_ax(axes[2])
    i6 = t6_idx(i)
    draw_composite(axes[0], xg6, yc6, u6_100.isel(time=i6).values,
                   xg24, yc24, u24_100.isel(time=i).values,
                   u100_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
    draw_composite(axes[1], xc6, yg6, v6_100.isel(time=i6).values,
                   xc24, yg24, v24_100.isel(time=i).values,
                   v100_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
    draw_composite(axes[2], xc6, yc6, w6_100.isel(time=i6).values,
                   xc24, yc24, w24_100.isel(time=i).values,
                   w100_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
    title_txt.set_text(f'U/V/W at {z_100:.0f}m  {str(t24_times[i])[:13]}  '
                       f'[TP6: {str(t6_times[i6])[:10]}]')

anim = animation.FuncAnimation(fig, draw_vel_100, frames=len(t24_times), interval=150)
print('Rendering 100m velocity video...')
anim.save('velocity/video_velocity_100m.mp4', writer=_writer, dpi=100)
plt.close()
print('Saved video_velocity_100m.mp4')
del u24_100, v24_100, w24_100

Loading TPOSE24 100 m U/V/W...


Saved snapshot_velocity_100m.png


Rendering 100m velocity video...


Saved video_velocity_100m.mp4


## 9. Velocity at 500 m

In [10]:
print('Loading TPOSE24 500 m U/V/W...')
z_500 = float(ds24.Z.sel(Z=-500., method='nearest'))
u24_500 = ds24_6h.UVEL.sel(Z=z_500, method='nearest').compute()
v24_500 = ds24_6h.VVEL.sel(Z=z_500, method='nearest').compute()

zl24_500 = float(ds24.Zl.sel(Zl=-500., method='nearest'))
w24_500 = ds24_6h.WVEL.sel(Zl=zl24_500, method='nearest').compute()

combined_u500 = np.concatenate([u24_500.values.ravel(), u6_500.values.ravel()])
combined_v500 = np.concatenate([v24_500.values.ravel(), v6_500.values.ravel()])
combined_w500 = np.concatenate([w24_500.values.ravel(), w6_500.values.ravel()])
u500_levels = make_levels(combined_u500, symmetric=True)
v500_levels = make_levels(combined_v500, symmetric=True)
w500_levels = make_levels(combined_w500, symmetric=True)

# --- Snapshot ---
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
panels = [
    (axes[0], u24_500.isel(time=snap_i24).values, u6_500.isel(time=snap_i6).values,
     xg24, yc24, xg6, yc6, u500_levels, f'UVEL (m/s) Z≈{z_500:.0f}m'),
    (axes[1], v24_500.isel(time=snap_i24).values, v6_500.isel(time=snap_i6).values,
     xc24, yg24, xc6, yg6, v500_levels, f'VVEL (m/s) Z≈{z_500:.0f}m'),
    (axes[2], w24_500.isel(time=snap_i24).values, w6_500.isel(time=snap_i6).values,
     xc24, yc24, xc6, yc6, w500_levels, f'WVEL (m/s) Zl≈{zl24_500:.0f}m'),
]
for ax, arr24, arr6, x24, y24, x6, y6, levels, label in panels:
    draw_composite(ax, x6, y6, arr6, x24, y24, arr24, levels, cmap_vel,
                   lon_min24, lon_max24, lat_min24, lat_max24)
    add_colorbar(fig, ax, levels, cmap_vel, label)
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
axes[0].set_title(f'U at {z_500:.0f} m  {str(t24_times[snap_i24])[:13]}')
axes[1].set_title(f'V at {z_500:.0f} m  {str(t24_times[snap_i24])[:13]}')
axes[2].set_title(f'W at Zl≈{zl24_500:.0f} m  {str(t24_times[snap_i24])[:13]}')
fig.suptitle(f'Velocity at Z≈{z_500:.0f}m: TPOSE6 background + TPOSE24 fill', fontsize=12)
plt.tight_layout()
plt.savefig('velocity/snapshot_velocity_500m.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved snapshot_velocity_500m.png')

# --- Animation ---
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for ax in axes:
    ax.set_xlim(buf_lon_min, buf_lon_max)
    ax.set_ylim(buf_lat_min, buf_lat_max)
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
add_colorbar(fig, axes[0], u500_levels, cmap_vel, f'UVEL (m/s) {z_500:.0f}m')
add_colorbar(fig, axes[1], v500_levels, cmap_vel, f'VVEL (m/s) {z_500:.0f}m')
add_colorbar(fig, axes[2], w500_levels, cmap_vel, f'WVEL (m/s) {zl24_500:.0f}m')
title_txt = fig.suptitle('', fontsize=11)
draw_composite(axes[0], xg6, yc6, u6_500.isel(time=0).values,
               xg24, yc24, u24_500.isel(time=0).values,
               u500_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
draw_composite(axes[1], xc6, yg6, v6_500.isel(time=0).values,
               xc24, yg24, v24_500.isel(time=0).values,
               v500_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
draw_composite(axes[2], xc6, yc6, w6_500.isel(time=0).values,
               xc24, yc24, w24_500.isel(time=0).values,
               w500_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
plt.tight_layout()

def draw_vel_500(i):
    clear_ax(axes[0]); clear_ax(axes[1]); clear_ax(axes[2])
    i6 = t6_idx(i)
    draw_composite(axes[0], xg6, yc6, u6_500.isel(time=i6).values,
                   xg24, yc24, u24_500.isel(time=i).values,
                   u500_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
    draw_composite(axes[1], xc6, yg6, v6_500.isel(time=i6).values,
                   xc24, yg24, v24_500.isel(time=i).values,
                   v500_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
    draw_composite(axes[2], xc6, yc6, w6_500.isel(time=i6).values,
                   xc24, yc24, w24_500.isel(time=i).values,
                   w500_levels, cmap_vel, lon_min24, lon_max24, lat_min24, lat_max24)
    title_txt.set_text(f'U/V/W at {z_500:.0f}m  {str(t24_times[i])[:13]}  '
                       f'[TP6: {str(t6_times[i6])[:10]}]')

anim = animation.FuncAnimation(fig, draw_vel_500, frames=len(t24_times), interval=150)
print('Rendering 500m velocity video...')
anim.save('velocity/video_velocity_500m.mp4', writer=_writer, dpi=100)
plt.close()
print('Saved video_velocity_500m.mp4')
del u24_500, v24_500, w24_500

print('\nAll outputs:')
import os
for f in sorted(os.listdir('.')):
    if f.startswith(('snapshot_', 'video_')) and not f.endswith('.html'):
        size_mb = os.path.getsize(f) / 1e6
        print(f'  {f:45s} {size_mb:.1f} MB')

Loading TPOSE24 500 m U/V/W...


Saved snapshot_velocity_500m.png


Rendering 500m velocity video...


Saved video_velocity_500m.mp4

All outputs:
  snapshot_SSH.png                              0.1 MB
  snapshot_SST.png                              0.2 MB
  snapshot_velocity_100m.png                    0.9 MB
  snapshot_velocity_500m.png                    1.0 MB
  snapshot_velocity_surface.png                 0.5 MB
  video_SSH.mp4                                 1.0 MB
  video_SST.mp4                                 0.9 MB
  video_velocity_100m.mp4                       8.7 MB
  video_velocity_500m.mp4                       9.4 MB
  video_velocity_surface.mp4                    3.7 MB
